# 6 · Time, repair and a simulation that checks the algebra

Availability and MTBF with repair, and the Monte Carlo checks. See
[Lesson 6](../docs/lessons/06-time-repair-and-a-simulation-that-checks-the-algebra.md).

In [1]:
import matplotlib.pyplot as plt
import numpy as np

from cyberccf import plotting
from cyberccf.independence_baseline import MISSION_HOURS, navigation_model, unreliability

plotting.use_style()

In [2]:
from cyberccf.cyber_ccf_model import cyber_ccf_model
from cyberccf.independence_baseline import component_specs
from cyberccf.physical_ccf_beta_factor import physical_ccf_model
from cyberccf.time_dependent import steady_state

mttr = {n: s.mttr_hours for n, s in component_specs(mission_computers=2).items()}
for label, model in [
    ("as claimed", navigation_model(mission_computers=2)),
    ("physical CCF", physical_ccf_model(mission_computers=2)),
    ("physical + cyber CCF", cyber_ccf_model(mission_computers=2)),
]:
    ss = steady_state(model, mttr)
    print(f"{label:22s} A {ss.availability:.7f}  MTBF {ss.mtbf:10.0f} h  MDT {ss.mdt:.2f} h")

as claimed             A 0.9999994  MTBF    2958862 h  MDT 1.67 h
physical CCF           A 0.9999865  MTBF     146494 h  MDT 1.98 h
physical + cyber CCF   A 0.9999269  MTBF      24486 h  MDT 1.79 h


In [3]:
from cyberccf.time_dependent import simulate_missions, transient_unreliability

model = cyber_ccf_model(mission_computers=2)
est = simulate_missions(model, 2000.0, 200_000, np.random.default_rng(1))
print(f"BDD {unreliability(model, 2000.0):.4f}  Markov {transient_unreliability(model, 2000.0):.4f}  "
      f"Monte Carlo {est.value:.4f} [{est.low:.4f}, {est.high:.4f}]")

BDD 0.1978  Markov 0.1978  Monte Carlo 0.1975 [0.1948, 0.2002]
